# 02 — Feature Extraction
Loads a split, extracts spectral + texture + index features from images, fits all transformers on train, applies to val, and saves processed features.

**Run `01_split.ipynb` first.**

In [ ]:
# ── CONFIG ── change this to switch splits
SPLIT    = 'holdout_time'   # 'stratified' | 'holdout_time'
IMG_SIZE = 128

In [2]:
import numpy as np
import pandas as pd
import tifffile
import cv2
import joblib
import itertools
from pathlib import Path
from datetime import datetime, timezone
from tqdm import tqdm
from scipy.stats import kurtosis, skew, entropy
from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import QuantileTransformer,StandardScaler
from pysolar.solar import get_azimuth

ROOT   = Path('D:/Regional-Scale-AOD-Estimation')
SPLITS = ROOT / 'feature_engineering/splits'
OUT    = ROOT / 'feature_engineering/features' / SPLIT
OUT.mkdir(parents=True, exist_ok=True)

BAND_NAMES = ['B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B10','B11','B12']
EPS = 1e-6

## Load split & prepare metadata

In [3]:
train_df = pd.read_csv(SPLITS / f'{SPLIT}_train.csv')
val_df   = pd.read_csv(SPLITS / f'{SPLIT}_val.csv')

def parse_date(s):
    d, m, y = s.split(':')
    return datetime(int(y), int(m), int(d))

def compute_saa(image_paths, lats, lons):
    """Solar Azimuth Angle at Sentinel-2 overpass time.

    The image filename encodes the Sentinel-2 overpass UTC datetime
    (format YYYYMMDD_HHMMSS), distinct from the AERONET measurement time
    stored in dataset.csv. Using the overpass time here gives accurate SAA
    (error < 0.1° vs using AERONET time which could be off by up to ~7.5°).
    """
    saas = []
    for p, lat, lon in zip(image_paths, lats, lons):
        fname = Path(p).stem  # e.g. '20190815_082031' — Sentinel-2 overpass time
        dt    = datetime.strptime(fname, "%Y%m%d_%H%M%S").replace(tzinfo=timezone.utc)
        saas.append(get_azimuth(float(lat), float(lon), dt))
    return np.array(saas, dtype=np.float32)

for df_ in [train_df, val_df]:
    dates = df_['Date(dd:mm:yyyy)'].apply(parse_date)
    df_['Year']  = dates.dt.year
    df_['Month'] = dates.dt.month
    sza_rad        = np.deg2rad(df_['Solar_Zenith_Angle(Degrees)'].to_numpy())
    assert (df_['Solar_Zenith_Angle(Degrees)'] < 80).all(), \
        "SZA ≥ 80° found — airmass will be extreme (>5.8); check dataset"
    df_['Airmass'] = 1.0 / np.cos(sza_rad)
    df_['Solar_Azimuth_Angle(Degrees)'] = compute_saa(
        df_['image_path'], df_['Site_Latitude(Degrees)'], df_['Site_Longitude(Degrees)']
    )

META_COLS  = ['Day_of_Year', 'Site_Latitude(Degrees)', 'Site_Longitude(Degrees)',
              'Site_Elevation(m)', 'Solar_Zenith_Angle(Degrees)',
              'Solar_Azimuth_Angle(Degrees)', 'Airmass', 'Year', 'Month']
TARGET_COL = 'AOD_550nm'
PATH_COL   = 'image_path'

train_meta = train_df[META_COLS + [TARGET_COL, PATH_COL]].reset_index(drop=True)
val_meta   = val_df[META_COLS + [TARGET_COL, PATH_COL]].reset_index(drop=True)

print(f'Train: {len(train_meta)} | Val: {len(val_meta)}')
print(f'SAA range — train: [{train_meta["Solar_Azimuth_Angle(Degrees)"].min():.1f}, '
      f'{train_meta["Solar_Azimuth_Angle(Degrees)"].max():.1f}]°')
train_meta.head(2)

Train: 5166 | Val: 1017
SAA range — train: [82.6, 166.4]°


,Day_of_Year,Site_Latitude(Degrees),Site_Longitude(Degrees),Site_Elevation(m),Solar_Zenith_Angle(Degrees),Solar_Azimuth_Angle(Degrees),Airmass,Year,Month,AOD_550nm,image_path
0,232,30.8550,34.782222,480.0,25.127998,133.531708,1.104531,2015,8,0.212580,Data_Collection/Dataset/images/SEDE_BOKER/2015...
1,232,29.5025,34.917500,15.0,24.136170,131.793991,1.095799,2015,8,0.302683,Data_Collection/Dataset/images/Eilat/20150820_...


## Load images

## Load images

### Cloud / nodata pixel handling — design rationale

Bad pixels (DN = 0 nodata, DN > 10 000 saturated/cloud) are replaced with the **per-band scene mean of valid pixels** before feature extraction. A known limitation of this approach is that mean-imputed pixels create artificial intensity boundaries with their neighbours, which can corrupt GLCM texture co-occurrences.

Two pipeline choices bound the severity of this effect:

1. **Coverage filter (`min_coverage = 0.90`)** — any image where more than 10 % of pixels in any band are invalid is dropped entirely. The imputation therefore operates on at most ~10 % of pixels, and in practice far fewer (the remaining bad pixels after the coverage drop are sparse nodata edges, not contiguous cloud).

2. **Percentile normalisation in `to_uint8_glcm()`** — GLCM patches are mapped to [0, 255] via the 1st–99th percentile of that patch. Because the imputed value *is* the scene mean, it sits near the centre of the valid-pixel distribution; the percentile mapping places it close to DN ≈ 127. A cloud pixel left un-imputed (DN > 10 000, reflectance > 1.0) would map to 255 and create a far sharper discontinuity. Mean imputation therefore **reduces** rather than inflates GLCM contrast compared with leaving cloud DNs in place.

The residual bias (artificial co-occurrences between imputed pixels and their valid neighbours) is accepted as a minor artefact, justified by the operational goal of retaining partially-observed scenes rather than discarding them.

---

### Spatial downsampling to 128 × 128 — design rationale

Images are resized from their native ~500 × 500 pixels (10 m resolution, ~5 km × 5 km patch) to `IMG_SIZE = 128 × 128` using `cv2.INTER_AREA`, which averages pixel blocks and is the geometrically correct choice for downsampling. The primary driver for this choice was **memory**. At native resolution, loading the full training set (~4 000 images × 13 bands × 500² pixels × 4 bytes) requires approximately 50 GB of RAM, which exceeds the available hardware. At 128 × 128, the same data fits within ~3.5 GB.

Two points address the concern that downsampling degrades GLCM:

1. **Scale of the target quantity.** AOD is a column-integrated atmospheric property measured by a sun photometer with an ~1.2° field of view, averaging over a footprint of roughly 5–10 km in diameter at flight altitude. The feature patch (5 km × 5 km) is already sized to match this footprint. Within that footprint, surface heterogeneity at the 10 m scale (individual trees, rooftops) carries negligible signal for a column quantity. After resampling, one GLCM pixel represents ~39 m — a scale that still captures meaningful land-cover texture (urban/rural, bare soil, sparse vegetation) while discarding sub-pixel noise.

2. **`INTER_AREA` preserves radiometric averages.** Unlike bilinear or bicubic interpolation, area-averaging does not introduce new spectral values; each output pixel is the mean of the corresponding input block. Band ratio indices computed from these values therefore remain physically meaningful. The GLCM distance-1 co-occurrence is reinterpreted as capturing texture at ~40 m rather than 10 m scale, which is consistent with the spatial scale of the retrieval target.

In [4]:
def load_bands(paths, img_size=IMG_SIZE, min_coverage=0.90, cirrus_p95_thresh=300.0):
    """Read images, resize, filter clouds & cirrus, fill bad pixels with per-band mean.

    Images stored as (H, W, 13) channels-last by tifffile.
    Good pixels: DN in (0, 10000]. DN==0 is nodata; DN>10000 is cloud/saturated.
    Images with <min_coverage good pixels in any band are dropped.
    Remaining bad pixels (nodata + cloud, ≤10% by construction) are replaced
    with the per-band mean of good pixels before feature extraction.

    Returns:
        bands_sqrt : (N_kept, 13, img_size²) float32 — sqrt(raw), for band statistics
        bands_raw  : (N_kept, 13, img_size²) float32 — raw/10000 reflectance, for physical indices
        kept_idx   : list of original indices that passed the filters
    """
    sqrt_out, raw_out, kept_idx = [], [], []
    n_dropped_coverage = 0
    n_dropped_cirrus   = 0

    for idx, p in enumerate(tqdm(paths, desc='loading')):
        img = tifffile.imread(str(ROOT / p)).astype(np.float32)  # (H, W, 13)
        img = np.transpose(img, (2, 0, 1))                       # (13, H, W)

        resized = np.stack([
            cv2.resize(img[b], (img_size, img_size), interpolation=cv2.INTER_AREA)
            for b in range(img.shape[0])
        ], axis=0)  # (13, img_size, img_size)

        n_pixels  = img_size * img_size
        good_mask = (resized > 0) & (resized <= 10000)

        # Coverage filter: drop if any band has <min_coverage good pixels
        if good_mask.sum(axis=(1, 2)).min() / n_pixels < min_coverage:
            n_dropped_coverage += 1
            continue

        # Cirrus screen on B10 (index 10)
        b10_good = resized[10][good_mask[10]]
        if b10_good.size == 0 or np.percentile(b10_good, 95) > cirrus_p95_thresh:
            n_dropped_cirrus += 1
            continue

        # Replace bad pixels (nodata + cloud, ≤10% by construction) with per-band good mean.
        # This prevents cloud/nodata DNs from distorting statistics and ratio indices.
        for b in range(resized.shape[0]):
            bad = ~good_mask[b]
            if bad.any():
                good_vals = resized[b][good_mask[b]]
                resized[b][bad] = good_vals.mean() if good_vals.size > 0 else 0.0

        sqrt_out.append(np.sqrt(resized).reshape(13, -1))
        raw_out.append((resized / 10000.0).reshape(13, -1))
        kept_idx.append(idx)

    print(f'  Dropped — coverage: {n_dropped_coverage} | cirrus: {n_dropped_cirrus} | kept: {len(kept_idx)}')
    return (np.asarray(sqrt_out, dtype=np.float32),
            np.asarray(raw_out,  dtype=np.float32),
            kept_idx)

print('Loading train images...')
n_train_total = len(train_meta)
train_bands_sqrt, train_bands_raw, train_kept = load_bands(train_meta[PATH_COL])
assert len(train_kept) > 0, "All train images were filtered out — check coverage/cirrus thresholds"
train_meta = train_meta.iloc[train_kept].reset_index(drop=True)
print(f'Train: {len(train_kept)}/{n_train_total} kept | shape: {train_bands_sqrt.shape}')

Loading train images...


loading: 100%|██████████| 5166/5166 [03:12<00:00, 26.87it/s]


  Dropped — coverage: 807 | cirrus: 59 | kept: 4300
Train: 4300/5166 kept | shape: (4300, 13, 16384)


In [5]:
print('Loading val images...')
n_val_total = len(val_meta)
val_bands_sqrt, val_bands_raw, val_kept = load_bands(val_meta[PATH_COL])
assert len(val_kept) > 0, "All val images were filtered out — check coverage/cirrus thresholds"
val_meta = val_meta.iloc[val_kept].reset_index(drop=True)
print(f'Val: {len(val_kept)}/{n_val_total} kept | shape: {val_bands_sqrt.shape}')

Loading val images...


loading: 100%|██████████| 1017/1017 [00:34<00:00, 29.39it/s]


  Dropped — coverage: 168 | cirrus: 15 | kept: 834
Val: 834/1017 kept | shape: (834, 13, 16384)


## Feature extraction functions

In [6]:
# ── helpers ────────────────────────────────────────────────────────────────

def to_uint8_glcm(patch_2d):
    """Normalise a 2D array to [0,255] uint8 for GLCM.
    Uses 1st–99th percentile clipping so a single extreme pixel (e.g. a cloud
    edge in a ratio index that can span several orders of magnitude) does not
    compress the entire dynamic range into a flat grey patch.
    """
    lo, hi = np.percentile(patch_2d, [1, 99])
    if hi - lo < EPS:
        return np.zeros_like(patch_2d, dtype=np.uint8)
    return np.clip((patch_2d - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)

def _stats(vals, prefix):
    """10 summary statistics for a (N, P) array. Returns dict."""
    return {
        f'{prefix}_mean'    : np.mean(vals, axis=1),
        f'{prefix}_var'     : np.var(vals, axis=1),
        f'{prefix}_median'  : np.median(vals, axis=1),
        f'{prefix}_kurtosis': kurtosis(vals, axis=1),
        f'{prefix}_skew'    : skew(vals, axis=1),
        f'{prefix}_std'     : np.std(vals, axis=1),
        f'{prefix}_p10'     : np.percentile(vals, 10, axis=1),
        f'{prefix}_p25'     : np.percentile(vals, 25, axis=1),
        f'{prefix}_p75'     : np.percentile(vals, 75, axis=1),
        f'{prefix}_p90'     : np.percentile(vals, 90, axis=1),
    }

def _glcm(vals, prefix, img_size=IMG_SIZE):
    """5 GLCM texture features for a (N, P) array. Returns dict."""
    props = ['dissimilarity', 'homogeneity', 'contrast', 'correlation', 'energy']
    cols  = {f'{prefix}_GLCM_{p}': [] for p in props}
    for i in range(vals.shape[0]):
        patch = to_uint8_glcm(vals[i].reshape(img_size, img_size))
        g     = graycomatrix(patch, distances=[1], angles=[0], symmetric=True, normed=True)
        for p in props:
            cols[f'{prefix}_GLCM_{p}'].append(graycoprops(g, p)[0, 0])
    return cols

def _index_features(index_dict, img_size=IMG_SIZE):
    """Stats + GLCM for every index in index_dict."""
    new_cols, glcm_cols = {}, {}
    for name, vals in tqdm(index_dict.items(), desc='  indices', leave=False):
        new_cols.update(_stats(vals, name))
        glcm_cols.update(_glcm(vals, name, img_size))
    return pd.DataFrame({**new_cols, **glcm_cols})


# ── band-level features (on bands_sqrt) ───────────────────────────────────

def calculate_statistics(bands, band_names=BAND_NAMES):
    """Per-band stats + all pairwise (Ba-Bb)/(Ba+Bb) and Ba/Bb ratio stats."""
    new_cols = {}
    for j, name in enumerate(tqdm(band_names, desc='band stats')):
        new_cols.update(_stats(bands[:, j, :], name))
    pairs = list(itertools.combinations(range(len(band_names)), 2))
    for (i, j) in tqdm(pairs, desc='band ratios'):
        Ba, Bb = bands[:, i, :], bands[:, j, :]
        new_cols.update(_stats((Ba - Bb) / (Ba + Bb + EPS), f'rdiff_{i}_{j}'))
        new_cols.update(_stats(Ba / (Bb + EPS),             f'rdiv_{i}_{j}'))
    return pd.DataFrame(new_cols)


def calculate_advanced_band_features(bands, band_names=BAND_NAMES):
    """Geometric/harmonic means, normalised ratio, per-band entropy, triple-band combos."""
    new_cols = {}
    pairs   = list(itertools.combinations(range(len(band_names)), 2))
    triples = list(itertools.combinations(range(len(band_names)), 3))

    for (i, j) in tqdm(pairs, desc='advanced pairs'):
        Ba, Bb = bands[:, i, :], bands[:, j, :]
        new_cols[f'geomean_{i}_{j}']   = np.sqrt(Ba * Bb).mean(axis=1)
        new_cols[f'harmmean_{i}_{j}']  = (2 * Ba * Bb / (Ba + Bb + EPS)).mean(axis=1)
        new_cols[f'normratio_{i}_{j}'] = ((Ba - Bb) / (np.sqrt(Ba**2 + Bb**2) + EPS)).mean(axis=1)

    for i, name in tqdm(enumerate(band_names), desc='entropy', total=len(band_names)):
        new_cols[f'{name}_entropy'] = np.apply_along_axis(
            lambda x: entropy(np.histogram(x, bins=256, density=True)[0] + EPS), 1, bands[:, i, :]
        )

    for (i, j, k) in tqdm(triples, desc='triple combos'):
        Ba, Bb, Bc = bands[:, i, :], bands[:, j, :], bands[:, k, :]
        new_cols[f'dprod_{i}_{j}_{k}']  = ((Ba * Bb) - (Ba * Bc)).mean(axis=1)
        new_cols[f'cratio_{i}_{j}_{k}'] = ((Ba * Bb) / (Bc + EPS)).mean(axis=1)

    return pd.DataFrame(new_cols)


def compute_texture_features(bands_raw, img_size=IMG_SIZE):
    """GLCM on B02 (blue) raw reflectance only, using a fixed scale.

    Fixed [0, 0.5 reflectance] → [0, 255] mapping preserves absolute contrast
    across images — hazy images have narrower histograms, clear images wider.
    4-direction averaging gives rotation invariance.
    """
    B02       = bands_raw[:, 1, :]
    scale_max = 0.5
    props     = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
    cols      = {f'B02raw_GLCM_{p}': [] for p in props}

    for i in tqdm(range(B02.shape[0]), desc='GLCM B02 raw'):
        patch = np.clip(B02[i], 0.0, scale_max) / scale_max
        patch = (patch * 255.0).astype(np.uint8).reshape(img_size, img_size)
        g     = graycomatrix(patch, distances=[1],
                             angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                             levels=256, symmetric=True, normed=True)
        for p in props:
            cols[f'B02raw_GLCM_{p}'].append(graycoprops(g, p).mean())

    return pd.DataFrame(cols)


# ── spectral index features (on bands_raw — true reflectance 0-1) ─────────
# Band layout: B01=0 B02=1 B03=2 B04=3 B05=4 B06=5 B07=6
#              B08=7 B8A=8 B09=9 B10=10 B11=11 B12=12

def calculate_vegetation_indices(bands):
    B01=bands[:,0,:];  B02=bands[:,1,:];  B03=bands[:,2,:]
    B04=bands[:,3,:];  B05=bands[:,4,:];  B06=bands[:,5,:]
    B07=bands[:,6,:];  B08=bands[:,7,:];  B8A=bands[:,8,:]
    B09=bands[:,9,:];  B10=bands[:,10,:]; B11=bands[:,11,:]; B12=bands[:,12,:]

    indices = {
        'ndvi'  : (B08 - B04) / (B08 + B04 + EPS),
        'gli'   : (2*B03 - B04 - B02) / (2*B03 + B04 + B02 + EPS),
        'cvi'   : (B08 * B04) / (B03**2 + EPS),
        'sipi'  : (B08 - B02) / (B08 - B04 + EPS),
        'ndwi'  : (B08 - B11) / (B08 + B11 + EPS),
        'ccci'  : (B08 - B05) / (B08 + B05 + EPS) / ((B08 - B04) / (B08 + B04) + EPS),
        'hue'   : np.arctan2((B02 - B03), (B03 - B04)),
        'rendvi': (B06 - B05) / (B06 + B05 + EPS),
        'reci'  : (B08 / (B04 + EPS)) - 1,
        'evi'   : 2.5 * (B08 - B04) / (B08 + 6*B04 - 7.5*B02 + 1.0 + EPS),
        'evi2'  : 2.4 * (B08 - B04) / (B08 + B04 + 1.0 + EPS),
        'bsi'   : (B11 + B04) / (B08 + B02 + EPS),
        'npcri' : (B03 - B08) / (B03 + B08 + EPS),
        'mndvi' : (B08 - B04) / (B08 + B04 - 2*B02 + EPS),
        'ndsi'  : (B03 - B11) / (B03 + B11 + EPS),
        'ndmi'  : (B08 - B11) / (B08 + B11 + EPS),
        'ndbi'  : (B11 - B08) / (B11 + B08 + EPS),
        'ndci'  : (B05 - B04) / (B05 + B04 + EPS),
        'gndvi' : (B08 - B03) / (B08 + B03 + EPS),
        'savi'  : ((B01 - B02) / (B01 + B02 + 0.428 + EPS)) * 1.428,
    }
    return _index_features(indices)


def calculate_vegetation_indices2(bands):
    B01=bands[:,0,:];  B02=bands[:,1,:];  B03=bands[:,2,:]
    B04=bands[:,3,:];  B05=bands[:,4,:];  B06=bands[:,5,:]
    B07=bands[:,6,:];  B08=bands[:,7,:];  B8A=bands[:,8,:]
    B09=bands[:,9,:];  B10=bands[:,10,:]; B11=bands[:,11,:]; B12=bands[:,12,:]

    indices = {
        'ndvi_r': (B08 - B07) / (B08 + B07 + EPS),
        'chl'   : (B07 / (B05 + EPS)) - 1,
        'bi'    : (B04**2 + B03**2 + B02**2) / 3,
        'si'    : (B04 - B02) / (B04 + B02 + EPS),
        'nmdi'  : (B08 - (B11 - B12)) / (B08 + (B11 - B12) + EPS),
        'wbi'   : B8A / (B09 + EPS),
        'msi'   : B11 / (B08 + EPS),
        'bsi1'  : ((B12 + B04) - (B8A + B02)) / ((B12 + B04) + (B8A + B02) + EPS),
        'bsi2'  : ((B11 + B04) - (B8A + B02)) / ((B11 + B04) + (B8A + B02) + EPS),
        'ndsi1' : (B11 - B8A) / (B11 + B8A + EPS),
        'ndsi2' : (B12 - B03) / (B12 + B03 + EPS),
        'bi2'   : B04 + B11 - B8A,
        'dbsi'  : (B12 - B03)/(B12 + B03 + EPS) - (B8A - B04)/(B8A + B04 + EPS),
        'mbi'   : ((B11 + B12 + B8A) / (B11 + B12 + B8A + EPS)) + 0.5,
        'r01'   : B01 / (B03 + EPS),
        'r02'   : B01 / (B05 + EPS),
        'r03'   : B11 / (B12 + EPS),
        'r04'   : B05 / (B04 + EPS),
        'mi'    : (B8A - B11) / (B8A + B11 + EPS),
        'mresr' : (B06 - B01) / (B05 - B01 + EPS),
        'psri'  : (B04 - B02) / (B06 + EPS),
        'tvi'   : (120*(B06 - B03) - 200*(B04 - B03)) / 2,
        'arvi'  : (B08 - 2*B04 + B02) / (B08 + 2*B04 + B02 + EPS),
        'exg'   : 2*B03 - B04 - B02,
        'aci'   : B08 * (B04 + B03),
    }
    return _index_features(indices)


def calculate_bloom_indices(bands):
    B02=bands[:,1,:]; B03=bands[:,2,:]
    B04=bands[:,3,:]; B08=bands[:,7,:]

    NDGI = (B04 - B03) / (B04 + B03 + EPS)
    DYI  =  B04 / (B03 + EPS)
    NDPI = (0.5*(B04 + B02) - B03) / (0.5*(B04 + B02) + B03 + EPS)
    NDYI = (0.5*(B04 + B03) - B02) / (0.5*(B04 + B03) + B02 + EPS)
    PEBI = NDPI / ((NDGI + 1) * B08 + EPS)
    YEBI = NDYI / ((NDGI + 1) * B08 + EPS)

    return _index_features({'NDGI': NDGI, 'DYI': DYI, 'NDPI': NDPI,
                             'PEBI': PEBI, 'NDYI': NDYI, 'YEBI': YEBI})


def calculate_rededge_indices(bands):
    B03=bands[:,2,:]; B04=bands[:,3,:]
    B05=bands[:,4,:]; B06=bands[:,5,:]
    B07=bands[:,6,:]; B08=bands[:,7,:]

    indices = {
        'NDVIre1': (B08-B05)/(B08+B05+EPS),
        'NDVIre2': (B08-B06)/(B08+B06+EPS),
        'NDVIre3': (B08-B07)/(B08+B07+EPS),
        'NDRE1'  : (B06-B05)/(B06+B05+EPS),
        'NDRE2'  : (B07-B05)/(B07+B05+EPS),
        'NDRE3'  : (B07-B06)/(B07+B06+EPS),
        'CIre1'  : (B08/(B05+EPS)) - 1,
        'CIre2'  : (B08/(B06+EPS)) - 1,
        'CIre3'  : (B08/(B07+EPS)) - 1,
        'MCARI1' : ((B05-B04) - 0.2*(B05-B03)) * (B05/(B04+EPS)),
        'MCARI2' : ((B06-B04) - 0.2*(B06-B03)) * (B06/(B04+EPS)),
        'MCARI3' : ((B07-B04) - 0.2*(B07-B03)) * (B07/(B04+EPS)),
        'TCARI1' : 3*((B05-B04) - 0.2*(B05-B03)) * (B05/(B04+EPS)),
        'TCARI2' : 3*((B06-B04) - 0.2*(B06-B03)) * (B06/(B04+EPS)),
        'TCARI3' : 3*((B07-B04) - 0.2*(B07-B03)) * (B07/(B04+EPS)),
        'MTCI1'  : (B06-B05)/(B05-B04+EPS),
        'MTCI2'  : (B07-B05)/(B05-B04+EPS),
        'MTCI3'  : (B07-B06)/(B06-B04+EPS),
    }
    return _index_features(indices)


def calculate_aod_indices(bands_raw, sza_deg):
    """AOD-physics-motivated features on raw reflectance (0-1)."""
    B01=bands_raw[:,0,:];  B02=bands_raw[:,1,:];  B03=bands_raw[:,2,:]
    B04=bands_raw[:,3,:];  B08=bands_raw[:,7,:]
    B11=bands_raw[:,11,:]; B12=bands_raw[:,12,:]

    indices = {
        'aod_B01_over_B11'      : B01 / (B11 + EPS),
        'aod_B02_over_B11'      : B02 / (B11 + EPS),
        'aod_B02_over_B12'      : B02 / (B12 + EPS),
        'aod_nd_B02_B11'        : (B02 - B11) / (B02 + B11 + EPS),
        'aod_nd_B01_B12'        : (B01 - B12) / (B01 + B12 + EPS),
        'aod_B01_over_B02'      : B01 / (B02 + EPS),
        'aod_B04_over_B12'      : B04 / (B12 + EPS),
        'aod_spectral_curv_B02' : np.log(B01 + EPS) + np.log(B04 + EPS) - 2*np.log(B02 + EPS),
        'aod_AFRI1600'          : (B08 - 0.66*B11) / (B08 + 0.66*B11 + EPS),
        'aod_NDVI_minus_AFRI'   : ((B08 - B04) / (B08 + B04 + EPS)
                                   - (B08 - 0.66*B11) / (B08 + 0.66*B11 + EPS)),
        'aod_dust_B04_minus_B02': B04 - B02,
        'aod_dust_nd_B04_B11'   : (B04 - B11) / (B04 + B11 + EPS),
        'aod_B02_airmass'       : B02 * (1.0 / np.cos(np.deg2rad(sza_deg)))[:, None],
        'aod_log_B02_airmass'   : np.log(B02 + EPS) * (1.0 / np.cos(np.deg2rad(sza_deg)))[:, None],
    }

    wl       = np.array([443.0, 490.0, 560.0, 665.0], dtype=np.float32)
    log_wl_c = np.log(wl) - np.log(wl).mean()
    denom    = (log_wl_c**2).sum()
    stacked  = np.stack([B01, B02, B03, B04], axis=1)
    log_rho  = np.log(np.clip(stacked, EPS, None))
    log_rho_c = log_rho - log_rho.mean(axis=1, keepdims=True)
    indices['aod_angstrom_alpha'] = (log_wl_c[None, :, None] * log_rho_c).sum(axis=1) / denom

    new_cols = {}
    for name, vals in tqdm(indices.items(), desc='aod indices', leave=False):
        new_cols.update(_stats(vals, name))
    return pd.DataFrame(new_cols)


def extract_all_features(meta_df, bands_sqrt, bands_raw, sza_deg):
    dfs = [
        meta_df.drop(columns=[PATH_COL]).reset_index(drop=True),
        calculate_statistics(bands_sqrt),
        calculate_advanced_band_features(bands_sqrt),
        compute_texture_features(bands_raw),
        calculate_vegetation_indices(bands_raw),
        calculate_vegetation_indices2(bands_raw),
        calculate_bloom_indices(bands_raw),
        calculate_rededge_indices(bands_raw),
        calculate_aod_indices(bands_raw, sza_deg),
    ]
    return pd.concat(dfs, axis=1)

## Extract features

In [7]:
print('Extracting train features...')
train_features = extract_all_features(
    train_meta, train_bands_sqrt, train_bands_raw,
    train_meta['Solar_Zenith_Angle(Degrees)'].to_numpy()
) 
print(f'Train features: {train_features.shape}')

Extracting train features...


  indices:  52%|█████▏    | 13/25 [02:19<02:07, 10.63s/it]C:\Users\LOQ\AppData\Local\Temp\ipykernel_13028\4001907375.py:20: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_kurtosis': kurtosis(vals, axis=1),
C:\Users\LOQ\AppData\Local\Temp\ipykernel_13028\4001907375.py:21: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_skew'    : skew(vals, axis=1),


Train features: (4300, 3709)


In [8]:
print('Extracting val features...')
val_features = extract_all_features(
    val_meta, val_bands_sqrt, val_bands_raw,
    val_meta['Solar_Zenith_Angle(Degrees)'].to_numpy()
)
print(f'Val features: {val_features.shape}')

Extracting val features...


  indices:  52%|█████▏    | 13/25 [00:26<00:24,  2.03s/it]C:\Users\LOQ\AppData\Local\Temp\ipykernel_13028\4001907375.py:20: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_kurtosis': kurtosis(vals, axis=1),
C:\Users\LOQ\AppData\Local\Temp\ipykernel_13028\4001907375.py:21: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_skew'    : skew(vals, axis=1),
                                                            

Val features: (834, 3709)


## Post-processing
All transformers are **fit on train only**, then applied to val.

In [9]:
TARGET = 'AOD_550nm'

train_y = train_features[TARGET].values
val_y   = val_features[TARGET].values

feat_cols = [c for c in train_features.columns if c != TARGET]

# ── 1. Strict column alignment ───────────────────────────────────────────────
extra_in_val = [c for c in val_features.columns if c not in train_features.columns and c != TARGET]
assert not extra_in_val, f"val has unexpected columns absent from train: {extra_in_val[:5]}"

missing_in_val = [c for c in feat_cols if c not in val_features.columns]
if missing_in_val:
    print(f'Warning: {len(missing_in_val)} train columns absent from val — NaN-filling')
    for col in missing_in_val:
        val_features[col] = np.nan

train_X = train_features[feat_cols].copy()
val_X   = val_features[feat_cols].copy()   # strict same order as train

# ── 2. Replace inf ────────────────────────────────────────────────────────────
train_X.replace([np.inf, -np.inf], np.nan, inplace=True)
val_X.replace([np.inf, -np.inf], np.nan, inplace=True)

# ── 3. Clip pathologically large finite values from signed-denominator indices
#       (sipi, mndvi, MTCI, ccci use denominators that can approach ±EPS, producing
#       per-pixel values up to ~1/EPS = 1e6; per-image spatial means can reach ~1e3).
#       These are not caught by replace([inf,-inf]) and bias train_means used for val
#       NaN imputation. 1e4 is well above any physical reflectance ratio (~10–50)
#       while catching the EPS-singularity outliers.
train_X = train_X.mask(train_X.abs() > 1e4)
val_X   = val_X.mask(val_X.abs() > 1e4)

# ── 4. Drop columns where >80% of TRAIN values are NaN ───────────────────────
thresh    = 0.2 * len(train_X)
keep_cols = train_X.columns[train_X.notna().sum() >= thresh].tolist()
train_X   = train_X[keep_cols]
val_X     = val_X[keep_cols]
print(f'Columns after NaN drop: {len(keep_cols)}')

# ── 5. Fill NaN with train means (pandas .mean() skips NaN) ──────────────────
train_means = train_X.mean()
all_nan_cols = train_means[train_means.isna()].index.tolist()
if all_nan_cols:
    print(f'Warning: {len(all_nan_cols)} columns are all-NaN in train — filling with 0')
    train_means = train_means.fillna(0.0)

train_X = train_X.fillna(train_means)
val_X   = val_X.fillna(train_means)

assert not train_X.isna().any().any(), "NaNs remain in train after imputation"
assert not val_X.isna().any().any(),   "NaNs remain in val after imputation"

train_X_arr = train_X.to_numpy(dtype=np.float64)
val_X_arr   = val_X.to_numpy(dtype=np.float64)

# ── 6. Drop near-zero variance columns — IMAGE features only ─────────────────
# META_COLS are kept unconditionally; variance filter applies only to image features
meta_feat_cols = [c for c in keep_cols if c in META_COLS]
img_feat_cols  = [c for c in keep_cols if c not in META_COLS]
meta_idx = np.array([keep_cols.index(c) for c in meta_feat_cols])
img_idx  = np.array([keep_cols.index(c) for c in img_feat_cols])

img_stds      = train_X_arr[:, img_idx].std(axis=0)
keep_img_mask = img_stds > 1e-5
kept_img_cols = [c for c, k in zip(img_feat_cols, keep_img_mask) if k]
kept_img_idx  = img_idx[keep_img_mask]
print(f'Image features after variance filter: {keep_img_mask.sum()}/{len(img_feat_cols)} kept')

# ── 7. QuantileTransformer on kept image features only ───────────────────────
qt = QuantileTransformer(output_distribution='normal', random_state=42)
train_X_arr[:, kept_img_idx] = qt.fit_transform(train_X_arr[:, kept_img_idx])
val_X_arr[:, kept_img_idx]   = qt.transform(val_X_arr[:, kept_img_idx])
print('QuantileTransformer done')

# ── 8. Assemble final arrays: meta (unscaled) + kept image cols (scaled) ─────
final_cols    = meta_feat_cols + kept_img_cols
all_final_idx = np.concatenate([meta_idx, kept_img_idx])

train_X_final = train_X_arr[:, all_final_idx]
val_X_final   = val_X_arr[:, all_final_idx]
print(f'Final feature count: {len(final_cols)}')

Columns after NaN drop: 3707
Image features after variance filter: 3691/3698 kept
QuantileTransformer done
Final feature count: 3700


## Save

In [10]:
train_out = pd.DataFrame(train_X_final, columns=final_cols)
train_out.insert(0, TARGET, train_y)
val_out   = pd.DataFrame(val_X_final, columns=final_cols)
val_out.insert(0, TARGET, val_y)

train_out.to_csv(OUT / 'train.csv', index=False)
val_out.to_csv(OUT / 'val.csv', index=False)

joblib.dump({
    'qt'             : qt,
    'keep_cols'      : keep_cols,       # columns after NaN drop (input to qt)
    'train_means'    : train_means,     # for NaN imputation at inference
    'kept_img_cols'  : kept_img_cols,   # image cols to apply qt.transform (rebuild idx by name)
    'meta_feat_cols' : meta_feat_cols,  # meta cols kept unscaled (rebuild idx by name)
    'final_cols'     : final_cols,      # final column order after assembly
}, OUT / 'transformers.joblib')

print(f'Saved to {OUT}')
print(f'  train.csv        : {train_out.shape}')
print(f'  val.csv          : {val_out.shape}')
print(f'  transformers.joblib')

Saved to D:\Regional-Scale-AOD-Estimation\feature_engineering\features\holdout_time
  train.csv        : (4300, 3701)
  val.csv          : (834, 3701)
  transformers.joblib


In [11]:
# Apply QuantileTransformer to META_COLS in the saved CSVs and update transformers.joblib
train_out = pd.read_csv(OUT / 'train.csv')
val_out   = pd.read_csv(OUT / 'val.csv')

saved_meta_cols = [c for c in META_COLS if c in train_out.columns]

qt_meta = QuantileTransformer(output_distribution='normal', random_state=42)
train_out[saved_meta_cols] = qt_meta.fit_transform(train_out[saved_meta_cols])
val_out[saved_meta_cols]   = qt_meta.transform(val_out[saved_meta_cols])

train_out.to_csv(OUT / 'train.csv', index=False)
val_out.to_csv(OUT / 'val.csv', index=False)

transformers = joblib.load(OUT / 'transformers.joblib')
transformers['qt_meta'] = qt_meta
transformers['meta_feat_cols'] = saved_meta_cols
joblib.dump(transformers, OUT / 'transformers.joblib')

print(f'Meta cols QT-scaled and CSVs overwritten.')
print(f'  train: {train_out.shape} | val: {val_out.shape}')
print(f'  Meta cols: {saved_meta_cols}')
print(f'  Train meta range: [{train_out[saved_meta_cols].values.min():.4f}, {train_out[saved_meta_cols].values.max():.4f}]')

Meta cols QT-scaled and CSVs overwritten.
  train: (4300, 3701) | val: (834, 3701)
  Meta cols: ['Day_of_Year', 'Site_Latitude(Degrees)', 'Site_Longitude(Degrees)', 'Site_Elevation(m)', 'Solar_Zenith_Angle(Degrees)', 'Solar_Azimuth_Angle(Degrees)', 'Airmass', 'Year', 'Month']
  Train meta range: [-5.1993, 5.1993]


In [12]:
import pandas as pd
ttt = pd.read_csv('D:/Regional-Scale-AOD-Estimation/feature_engineering/features/holdout_time/train.csv')
ttt.head()

,AOD_550nm,Day_of_Year,Site_Latitude(Degrees),Site_Longitude(Degrees),Site_Elevation(m),Solar_Zenith_Angle(Degrees),Solar_Azimuth_Angle(Degrees),Airmass,Year,Month,...,aod_angstrom_alpha_mean,aod_angstrom_alpha_var,aod_angstrom_alpha_median,aod_angstrom_alpha_kurtosis,aod_angstrom_alpha_skew,aod_angstrom_alpha_std,aod_angstrom_alpha_p10,aod_angstrom_alpha_p25,aod_angstrom_alpha_p75,aod_angstrom_alpha_p90
0,0.212580,0.312376,0.006273,-0.196979,1.111702,-0.306940,-0.372334,-0.306940,-5.199338,0.284828,...,1.320311,-1.286737,1.320883,1.333857,-1.152565,-1.286737,1.292524,1.219530,1.352423,1.350600
1,0.302683,0.312376,-0.252311,0.101796,-0.851675,-0.376319,-0.436077,-0.376319,-5.199338,0.284828,...,-0.628507,0.929205,-1.688756,-0.983197,1.776661,0.929207,-0.622692,-0.796158,0.115555,0.265819
2,0.381347,0.338834,-0.148584,-1.817968,-0.072829,-0.299809,-0.413514,-0.299810,-5.199338,0.284828,...,0.002243,-0.895863,-0.096267,0.270839,0.184245,-0.895863,0.185794,0.104140,-0.409588,-0.461169
3,0.081462,0.411534,0.006273,-0.196979,1.111702,-0.145256,-0.169084,-0.145257,-5.199338,0.284828,...,1.372549,-0.950817,1.432723,1.520902,-1.393131,-0.950816,1.269648,1.246667,1.458398,1.506358
4,0.058300,0.411534,-0.252311,0.101796,-0.851675,-0.206927,-0.213628,-0.206929,-5.199338,0.284828,...,-0.942484,1.371124,-2.121399,-1.125954,1.634538,1.371126,-0.960200,-1.042129,0.315011,0.482575
